# 2. Estadísticas descriptivas — cohorte unificada

Caracteriza la cohorte de sujetos sanos antes de modelarla: conteos por base, distribución de fases, demografía (edad y sexo) y una primera **matriz de transición global**, que ya anticipa la estructura no aleatoria que el análisis de Markov estudia a fondo.

**Entradas:** los dos CSV del notebook 1 — **dataset/epocas_unificado.csv** y **dataset/pacientes_unificado.csv**.
**Salidas:** carpeta **estadisticas/** con las tablas (CSV) y las figuras (PNG).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path("..").resolve()
DATASET = RAIZ / "dataset"
OUT = RAIZ / "estadisticas"
OUT.mkdir(parents=True, exist_ok=True)

epocas = pd.read_csv(DATASET / "epocas_unificado.csv")
pacientes = pd.read_csv(DATASET / "pacientes_unificado.csv")
print("Épocas:", len(epocas), "| Pacientes:", len(pacientes))

Épocas: 112809 | Pacientes: 127


## 1. Resumen general por dataset

In [2]:
resumen = pacientes.groupby('dataset').agg(
    n_pacientes=('paciente', 'count'),
    n_epocas=('n_epocas', 'sum'),
    horas_total=('duracion_h', 'sum'),
    horas_media=('duracion_h', 'mean'),
    horas_min=('duracion_h', 'min'),
    horas_max=('duracion_h', 'max'),
    edad_media=('edad', 'mean'),
    edad_min=('edad', 'min'),
    edad_max=('edad', 'max'),
    n_mujeres=('sexo', lambda s: (s == 'F').sum()),
    n_hombres=('sexo', lambda s: (s == 'M').sum()),
).round(2)

# Fila TOTAL: toda la cohorte agregada
total = pd.DataFrame({
    'n_pacientes': [pacientes['paciente'].nunique()],
    'n_epocas': [pacientes['n_epocas'].sum()],
    'horas_total': [pacientes['duracion_h'].sum().round(2)],
    'horas_media': [pacientes['duracion_h'].mean().round(2)],
    'horas_min': [pacientes['duracion_h'].min()],
    'horas_max': [pacientes['duracion_h'].max()],
    'edad_media': [pacientes['edad'].mean().round(2)],
    'edad_min': [pacientes['edad'].min()],
    'edad_max': [pacientes['edad'].max()],
    'n_mujeres': [(pacientes['sexo'] == 'F').sum()],
    'n_hombres': [(pacientes['sexo'] == 'M').sum()],
}, index=['TOTAL'])
resumen = pd.concat([resumen, total])
resumen.to_csv(OUT / "resumen_general.csv")
resumen

,n_pacientes,n_epocas,horas_total,horas_media,horas_min,horas_max,edad_media,edad_min,edad_max,n_mujeres,n_hombres
CAP,16,15947,132.88,8.30,7.10,9.50,32.19,23.0,42.0,9,7
EDF,82,70410,586.78,7.16,3.68,13.66,41.88,25.0,60.0,46,36
SCO,29,26452,220.44,7.60,6.32,10.16,NaN,NaN,NaN,0,0
TOTAL,127,112809,940.10,7.40,3.68,13.66,40.30,23.0,60.0,55,43


## 2. Distribución de fases por dataset (porcentaje y conteo)

In [3]:
FASES = ['Vigilia', 'S1', 'S2', 'SWS', 'REM', 'Sin_clasificar']
conteos = (epocas.groupby(['dataset', 'fase_texto']).size()
           .unstack(fill_value=0).reindex(columns=FASES, fill_value=0))
porcent = (conteos.div(conteos.sum(axis=1), axis=0) * 100).round(2)
conteos.to_csv(OUT / "conteo_fases_por_dataset.csv")
porcent.to_csv(OUT / "porcentaje_fases_por_dataset.csv")

# Barras agrupadas: una barra por dataset en cada fase
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(FASES)); ancho = 0.25
for i, ds in enumerate(porcent.index):
    ax.bar(x + (i - 1) * ancho, porcent.loc[ds], ancho, label=ds)
ax.set_xticks(x); ax.set_xticklabels(FASES)
ax.set_ylabel('% de épocas'); ax.set_title('Distribución de fases por dataset')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "distribucion_fases.png", dpi=300); plt.close(fig)
porcent

fase_texto,Vigilia,S1,S2,SWS,REM,Sin_clasificar
dataset,,,,,,
CAP,9.39,2.90,40.67,24.26,22.74,0.04
EDF,7.70,7.91,51.67,12.06,20.53,0.12
SCO,8.93,7.05,40.87,19.48,17.01,6.67


## 3. Histograma de duración de los registros

In [4]:
fig, ax = plt.subplots(figsize=(10, 5))
for ds, gds in pacientes.groupby('dataset'):
    ax.hist(gds['duracion_h'], bins=15, alpha=0.55, label=f"{ds} (n={len(gds)})")
ax.set_xlabel('Horas de registro'); ax.set_ylabel('N pacientes')
ax.set_title('Duración de registros por dataset')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "duracion_histograma.png", dpi=300); plt.close(fig)
print('OK duracion_histograma.png')

OK duracion_histograma.png


## 4. Histograma de edades (CAP y EDF)

In [5]:
fig, ax = plt.subplots(figsize=(10, 5))
for ds, gds in pacientes.dropna(subset=['edad']).groupby('dataset'):
    ax.hist(gds['edad'], bins=12, alpha=0.55, label=f"{ds} (n={len(gds)})")
ax.set_xlabel('Edad (años)'); ax.set_ylabel('N pacientes')
ax.set_title('Distribución de edades por dataset')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "edad_histograma.png", dpi=300); plt.close(fig)
print('OK edad_histograma.png')

OK edad_histograma.png


## 5. Distribución por sexo

In [6]:
sexo = (pacientes.fillna({'sexo': 'NA'}).groupby(['dataset', 'sexo']).size()
        .unstack(fill_value=0))
sexo.to_csv(OUT / "sexo_por_dataset.csv")
fig, ax = plt.subplots(figsize=(8, 5))
sexo.plot(kind='bar', stacked=True, ax=ax, color={'F': '#ff8fa3', 'M': '#7eb6ff', 'NA': '#cccccc'})
ax.set_ylabel('N pacientes'); ax.set_title('Distribución por sexo')
ax.grid(axis='y', alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / "sexo_barras.png", dpi=300); plt.close(fig)
sexo

sexo,F,M,NA
dataset,,,
CAP,9,7,0
EDF,46,36,0
SCO,0,0,29


## 6. Matriz de transición global (todos los pacientes)

In [7]:
FASES5 = ['Vigilia', 'S1', 'S2', 'SWS', 'REM']
df5 = epocas[epocas['fase_num'].between(0, 4)].copy()  # solo las 5 fases fisiológicas (descarta 'Sin_clasificar')
N = 5
M = np.zeros((N, N))
# Cuenta cada par consecutivo (origen -> destino) dentro de cada paciente
for _, g in df5.groupby('paciente'):
    seq = g.sort_values('epoca')['fase_num'].to_numpy()
    for a, b in zip(seq[:-1], seq[1:]):
        M[int(a), int(b)] += 1
Pglobal = M / M.sum()                       # conjunta: cada celda sobre el total (toda la matriz suma 1)
Prengl  = M / M.sum(axis=1, keepdims=True)  # condicional P(destino|origen): cada renglón suma 1

pd.DataFrame(Pglobal, index=FASES5, columns=FASES5).round(6).to_csv(OUT / "transiciones_global.csv")
pd.DataFrame(Prengl, index=FASES5, columns=FASES5).round(6).to_csv(OUT / "transiciones_renglon.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, P, ttl in zip(axes, [Pglobal, Prengl], ['Global (suma=1)', 'Por renglón']):
    im = ax.imshow(P, cmap='Blues')
    for i in range(N):
        for j in range(N):
            v = P[i, j]
            t = '0' if v == 0 else (f'<0.001' if v < 0.001 else f'{v:.3f}')
            ax.text(j, i, t, ha='center', va='center', fontsize=9,
                    color='white' if v > P.max()*0.6 else 'black')
    ax.set_xticks(range(N)); ax.set_xticklabels(FASES5)
    ax.set_yticks(range(N)); ax.set_yticklabels(FASES5)
    ax.set_xlabel('Destino'); ax.set_ylabel('Origen'); ax.set_title(ttl)
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle('Matriz de transición unificada — todos los datasets')
fig.tight_layout(); fig.savefig(OUT / "matriz_transicion_unificada.png", dpi=300); plt.close(fig)
print('OK matriz_transicion_unificada.png')

OK matriz_transicion_unificada.png


## 7. Archivos exportados

In [8]:
archivos = sorted(OUT.glob('*'))
for a in archivos:
    tam = a.stat().st_size
    tam_s = f'{tam/1e3:.1f} KB' if tam < 1e6 else f'{tam/1e6:.2f} MB'
    print(f'  {a.name:50s} {tam_s}')

  conteo_fases_por_dataset.csv                       0.1 KB
  deslizador_metricas.csv                            0.4 KB
  deslizador_paso_a_paso.csv                         0.6 KB
  deslizador_tokens_auditoria.csv                    0.5 KB
  deslizador_trayectorias.csv                        0.9 KB
  diagnostico_ciclos_cobertura.csv                   0.5 KB
  diagnostico_ciclos_cobertura.png                   89.6 KB
  diagnostico_ciclos_histograma.png                  36.3 KB
  diagnostico_ciclos_por_paciente.csv                4.4 KB
  diagnostico_ciclos_resumen.csv                     0.2 KB
  distribucion_fases.png                             84.5 KB
  duracion_histograma.png                            88.9 KB
  duracion_s2_bloques.csv                            0.2 KB
  edad_histograma.png                                86.3 KB
  matrices_ciclo_diagonal.csv                        3.4 KB
  matrices_ciclo_s2_destinos.csv                     0.6 KB
  matriz_transicion_unificada.png  